# Install dependencies


In [9]:
# Install dependencies
%pip install anthropic python-dotenv


  Using cached anthropic-0.109.1-py3-none-any.whl.metadata (3.2 kB)
  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached docstring_parser-0.18.0-py3-none-any.whl.metadata (3.5 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
Using cached anthropic-0.109.1-py3-none-any.whl (923 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached docstring_parser-0.18.0-py3-none-any.whl (22 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
Using cached pydantic-2.13.4-py3-none-any.whl (472 kB

In [1]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-6"


In [3]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return message.content[0].text


In [14]:
# Make a starting list of messages
messages = []

# Add in the initial user question of "Define quantum computing in one sentence"
add_user_message(messages, "Define quantum computing in one sentence")

# messages

# Pass the list of messages into 'chat' to get an answer
answer = chat(messages)

# answer

# Take the answer and add it as an assistant message into our list
add_assistant_message(messages, answer)

# messages

# Add in the user's follow-up question
add_user_message(messages, "Write another sentence")

# Call chat again with the list of messages to get a final answer
answer = chat(messages)

answer

'Unlike classical computers that use bits representing either 0 or 1, quantum computers use quantum bits, or **qubits**, which can exist in multiple states simultaneously, enabling them to perform many calculations at once.'

In [12]:
messages = []

# Use a 'while True' loop to run the chatbot forever
while True:
    # Get user input
    user_input = input("> ")
    
    # Add user input to the list of messages
    add_user_message(messages, user_input)
    # Call Claude with the chat function
    answer = chat(messages)
    # Add generated text to the list of messages
    add_assistant_message(messages, answer)
    # Print the generated text
    print("---")
    print(answer)
    print("---")
    

Message(id='msg_018T9HiahKv5XMniJoWRHPpa', container=None, content=[TextBlock(citations=None, text='3 + 5 = **8**', type='text')], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=14, output_tokens=14, output_tokens_details=None, server_tool_use=None, service_tier='standard'))
---
3 + 5 = **8**
---


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: user messages must have non-empty content'}, 'request_id': 'req_011Cc5bysuPE5kxnFHxwVC7q'}

In [10]:
def chat_system_prompt(messages, system_prompt=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system_prompt:
        params["system"] = system_prompt

    message = client.messages.create(**params)
    
    return message.content[0].text


In [12]:
messages = []

system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

add_user_message(messages, "How do I solve 5x+3=2 for x?")
answer = chat_system_prompt(messages, system_prompt)

answer

"Let's work through this step by step!\n\nFirst, let's think about our goal - we want to get **x by itself** on one side of the equation.\n\n**Step 1:** Right now we have +3 on the left side with the x term. What could you do to both sides of the equation to start removing that +3?"

In [14]:
messages = []

system_prompt = "You are a Python engineer who writes very concise code"

add_user_message(
    messages,
    "Write a Python function that checks a string for duplicate characters."
)

answer = chat_system_prompt(messages, system_prompt)

answer

'```python\ndef has_duplicates(s: str) -> bool:\n    return len(s) != len(set(s))\n```\n\n**Examples:**\n```python\nprint(has_duplicates("hello"))   # True  (\'l\' appears twice)\nprint(has_duplicates("world"))   # False\nprint(has_duplicates("python"))  # False\nprint(has_duplicates("abcabc"))  # True\n```\n\nIf you also want to **identify which characters are duplicated**:\n\n```python\ndef find_duplicates(s: str) -> set:\n    seen = set()\n    return {c for c in s if c in seen or seen.add(c)}\n\nprint(find_duplicates("hello"))   # {\'l\'}\nprint(find_duplicates("abcabc"))  # {\'a\', \'b\', \'c\'}\n```'